<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/t04_MD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecular Dynamics simulation with GROMACS

The aim of this experience is to set up, run and analyse a Molecular Dynamics (MD) simulation of a protein in water. We will work with hen egg-white lysozyme and go through all the steps of a typical simulation: building the system, energy minimisation, equilibration, production and analysis.

The reference lectures are:

*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/02_StochasticMolecules.pdf
*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/03_MolecularDynamics.pdf

## Preliminary setup

MD simulations are much faster on a GPU, so before starting check that you have access to one:

In [ ]:
!nvidia-smi

If the above command is not found, then select `Runtime -> Change runtime type` and choose T4 GPU. Once it is ready, run the cell above again.

### Installing GROMACS

We install a ready-made version of GROMACS (2026.3, with GPU support) from [conda-forge](https://conda-forge.org) using `micromamba`, a small package manager. Nothing needs to be compiled, but about 2.5 GB are downloaded, so the installation takes a few minutes.

Colab forgets everything you installed when the session ends: **run this cell again every time you start a new session.**

In [ ]:
%%bash
# install GROMACS 2026.3 with CUDA (GPU) support from conda-forge, unless it is already installed
if ! command -v gmx > /dev/null; then
  cd /content
  curl -sL -o micromamba https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64
  chmod +x micromamba
  ./micromamba create -y -q -r /content/mamba -p /content/gmx -c conda-forge "gromacs=2026.3=nompi_cuda*"
  ln -sf /content/gmx/bin/gmx /usr/local/bin/gmx
fi
gmx --version | grep -E "GROMACS version|GPU support"

You should see `GROMACS version: 2026.3-conda_forge` and `GPU support: CUDA`.

**Only if the installation above fails**, you can compile GROMACS from the source code instead. This takes much longer (tens of minutes), so start it well before the tutorial and keep the browser tab active:

```
%%bash
apt-get update -qq && apt-get install -y -qq cmake build-essential libfftw3-dev ninja-build
cd /content
wget -q https://ftp.gromacs.org/gromacs/gromacs-2026.4.tar.gz
tar -xzf gromacs-2026.4.tar.gz
mkdir -p gromacs-2026.4/build && cd gromacs-2026.4/build
# CMAKE_CUDA_ARCHITECTURES=native compiles the GPU code only for the GPU you are using,
# which is much faster than the default (all GPU generations)
cmake -G Ninja .. -DCMAKE_BUILD_TYPE=Release -DGMX_GPU=CUDA -DCMAKE_CUDA_ARCHITECTURES=native
ninja -j$(nproc) && ninja install
ln -sf /usr/local/gromacs/bin/gmx /usr/local/bin/gmx
```

We also need `py3Dmol` to look at the structures and trajectories directly in the notebook:

In [ ]:
!pip install -q py3Dmol

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive

Make a new folder for this task using the menus on the left side: for example in `/drive/MyDrive/Structural_Bioinformatics` make a folder named `Task4`.

In [ ]:
# set the path to your local task folder
local_path = '/content/drive/MyDrive/Structural_Bioinformatics/Task4'
%cd {local_path}

In [ ]:
# get a copy of the GitHub structural bioinformatics repository (or update it if you already have it)
# it contains the input files for the simulations
!if [ -d ../sb ]; then (cd ../sb && git pull); else git clone --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git ../sb; fi

GROMACS writes its results as `.xvg` text files: a header (lines starting with `#` or `@`) followed by columns of numbers. The first column is the x axis (e.g. time), the others are the quantities you asked for. This small function reads them; we will use it for all the plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def read_xvg(filename):
    """Read a GROMACS .xvg file.
    Returns (data, legends): data is a 2D numpy array whose first column is the x axis,
    legends is the list of names of the other columns (it can be empty)."""
    rows, legends = [], []
    with open(filename) as f:
        for line in f:
            if line.startswith('@') and 'legend "' in line and line.split()[1].startswith('s'):
                legends.append(line.split('"')[1])
            elif not line.startswith(('#', '@')) and line.strip():
                rows.append([float(x) for x in line.split()])
    return np.array(rows), legends

## GROMACS basics

All GROMACS tools are modules of a single program named `gmx`. To get help about any module you can use either of

```
gmx help (module)
gmx (module) -h
```

where `(module)` is the name of the tool, e.g. `gmx solvate -h`. The list of all modules is printed by `gmx help commands` and is also available [online](https://manual.gromacs.org/2026.4/user-guide/cmdline.html).

Many GROMACS tools ask questions interactively, for example which group of atoms to use. In Colab we cannot type the answers, so we send them to the program with `printf` or `echo`, e.g. `echo Protein | gmx ...`. You can use the name of a group (`Protein`, `Backbone`, `C-alpha`, `SOL`, ...) or its number; names are safer because the numbers change from system to system.

### Units

The basic units in GROMACS are
```
Time: ps
Length: nm
Energy: kJ/mol
```

## Molecular Dynamics of lysozyme in solution

Here you will set up a simulation of hen egg-white lysozyme, a protein widely studied in structural biology. Its crystal structure is available in the PDB as 1AKI.

### Building the system

In [ ]:
# download the structure from the PDB (only if it is not already in the folder)
%cd {local_path}
!wget -nc -q https://files.rcsb.org/download/1AKI.pdb
!grep -c HOH 1AKI.pdb

The file contains the protein and the water molecules seen in the crystal (`HOH`). We remove the crystal waters, since the protein will be immersed in a box of water anyway:

In [ ]:
!grep -v HOH 1AKI.pdb > 1AKI_clean.pdb

Now we select the force field and the water model and build the topology of the system, that is the file describing all the interactions present in our molecule. In GROMACS this is done with `pdb2gmx`. Here we use the CHARMM27 force field with the TIP3P water model. `-ignh` tells `pdb2gmx` to ignore any hydrogen in the PDB file and to add them itself; protonation states are assigned for pH 7.

In [ ]:
!gmx pdb2gmx -f 1AKI_clean.pdb -o 1AKI_processed.gro -ignh -water tip3p -ff charmm27
# keep a copy of the topology of the protein alone (see below why)
!cp topol.top topol_protein.top

In [ ]:
# some new files have been created, in particular:
# topol.top is the TOPOLOGY file
# posre.itp contains position restraints for the protein heavy atoms (used during equilibration)
# 1AKI_processed.gro is the structure in the GROMACS format, now including hydrogen atoms
%ls

The next step is to place the protein in a box. The box must be large enough to avoid that the protein interacts with its own periodic images through the sides, but as small as possible to save computer time. We use `editconf` to build a rhombic dodecahedron, the box shape with the smallest volume for a given distance between periodic images:

In [ ]:
# rhombic dodecahedron box with at least 1.2 nm between the protein and the box sides
!gmx editconf -f 1AKI_processed.gro -bt dodecahedron -d 1.2 -o 1AKI_box.gro

Then we fill the box with water using `solvate`. `solvate` also adds the number of water molecules to the topology file. If you run the cell twice, a second line would be added: for this reason the cell always starts again from the topology of the protein alone, so that it is safe to run it more than once.

In [ ]:
!cp topol_protein.top topol.top
# spc216.gro contains an equilibrated box of 3-site water, it can be used for TIP3P
!gmx solvate -cp 1AKI_box.gro -cs spc216.gro -p topol.top -o 1AKI_solv.gro
!cp topol.top topol_solv.top

The output tells you how many water molecules have been added (several thousand). Look also at the end of `topol.top`: the `[ molecules ]` section lists what is in the system.

In [ ]:
!tail -5 topol.top

Lysozyme carries a net charge, and we also want a physiological salt concentration. To add ions, `genion` replaces some water molecules with Na$^+$ and Cl$^-$. It needs a GROMACS run input file (`.tpr`), which `grompp` builds by combining the structure, the topology and a file of MD parameters (`.mdp`).

`grompp` will stop with a warning because the system is not neutral yet. This is exactly what we are going to fix, so we allow one warning with `-maxwarn 1`. Do not use `-maxwarn` in other situations without reading the warning first!

`genion` asks which group of molecules can be replaced by ions: we answer `SOL`, the water. As before, the cell restarts from the topology without ions, so it can be run again safely.

In [ ]:
# copy an .mdp file from the downloaded structural bioinformatics repository
%cp ../sb/Data/md/mdps/0-em-steep.mdp .
!cp topol_solv.top topol.top
# prepare a GROMACS run input file combining structure, topology and parameters
!gmx grompp -f 0-em-steep.mdp -c 1AKI_solv.gro -p topol.top -o ions.tpr -maxwarn 1
# replace water molecules (SOL) with ions to neutralise the system and reach 100 mM NaCl
!echo SOL | gmx genion -s ions.tpr -conc 0.1 -neutral -p topol.top -o 1AKI_ions.gro

From the output you should see how many Na and Cl ions have been added.

**Question:** lysozyme has a net charge of +8 at pH 7. Can you see it in the number of ions that were added?

### Energy minimisation

Before starting an MD simulation the system must be in a low-energy configuration: atoms that are too close produce very large forces that would make the simulation unstable. We minimise the energy with the steepest descent algorithm:

In [ ]:
!gmx grompp -f 0-em-steep.mdp -c 1AKI_ions.gro -p topol.top -o em.tpr

In [ ]:
!gmx mdrun -deffnm em -v

The minimisation takes less than a minute and stops once the largest force on any atom is below 1000 kJ mol$^{-1}$ nm$^{-1}$. The final configuration is written to `em.gro`. We can follow how the potential energy decreased with `gmx energy`, which extracts quantities from the energy file (`.edr`):

In [ ]:
!echo Potential | gmx energy -f em.edr -o em_potential.xvg

data, legends = read_xvg('em_potential.xvg')
plt.figure(figsize=(8, 4))
plt.plot(data[:, 0], data[:, 1])
plt.xlabel('Minimisation step')
plt.ylabel('Potential energy (kJ/mol)')
plt.grid(True)
plt.show()

### Equilibration

Now we need to bring the system to the desired temperature and pressure. We run a first short MD simulation in which the protein heavy atoms are restrained to their starting positions (position restraints, the `posre.itp` file), while water and ions adapt to the protein and reach the correct temperature and density.

In [ ]:
%cd {local_path}
%cp ../sb/Data/md/mdps/3-pr-npt.mdp .
%cat 3-pr-npt.mdp

The most important parameters are:

```
define                   = -DPOSRES ; switch on the position restraints
integrator               = md       ; we do MD
dt                       = 0.002    ; with a 0.002 ps time step
nsteps                   = 25000    ; for 25000 x 0.002 = 50 ps (this is too short for real cases)
tcoupl                   = v-rescale; thermostat
ref_t                    = 298      ; at 298 K (25 C)
pcoupl                   = c-rescale; barostat
ref_p                    = 1.0      ; at 1 bar
gen_vel                  = yes      ; start from random velocities ...
gen_seed                 = -1       ; ... with a different random seed every time
```

Because of `gen_seed = -1`, every time you run this step the atoms start with different random velocities: two runs started in this way are independent trajectories of the same system. We will use this in the report to run replicas.

A full explanation of all parameters is available here: https://manual.gromacs.org/2026.4/user-guide/mdp-options.html

In [ ]:
# -r em.gro: reference positions for the position restraints
!gmx grompp -f 3-pr-npt.mdp -c em.gro -r em.gro -p topol.top -o relax.tpr

In [ ]:
# ready to run it
!gmx mdrun -deffnm relax -v

#### Did the equilibration work?

An MD simulation is only meaningful if the system is equilibrated. Let's check that temperature, pressure and density have reached stable values:

In [ ]:
!printf "Temperature\nPressure\nDensity\n0\n" | gmx energy -f relax.edr -o relax_energy.xvg

data, legends = read_xvg('relax_energy.xvg')
units = ['K', 'bar', 'kg/m$^3$']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, ax in enumerate(axes):
    y = data[:, i + 1]
    ax.plot(data[:, 0], y)
    ax.axhline(y.mean(), color='k', ls='--')
    ax.set_xlabel('Time (ps)')
    ax.set_ylabel(f'{legends[i]} ({units[i]})')
    ax.set_title(f'{legends[i]}: {y.mean():.1f} ± {y.std():.1f} {units[i]}')
plt.tight_layout()
plt.show()

**Questions:**
1. Is the average temperature close to the 298 K we asked for? How large are its fluctuations?
2. The pressure fluctuates by hundreds of bar around the target of 1 bar. Is this a problem? (Hint: think about how many atoms are in the box and about what pressure is, microscopically.)
3. Is the density close to that of water?

#### How fast is the simulation?

At the end of the log file GROMACS reports its performance in ns/day, i.e. how many nanoseconds of simulation it would produce in one day. You can also check that the GPU was used.

In [ ]:
# if the GPU was used, the log file says which tasks ran on it (PP = particle-particle forces, PME = long-range electrostatics)
!grep -A2 "GPU selected" relax.log
!grep -B1 -A1 "Performance:" relax.log

In [ ]:
# estimate how long the production run will take
with open('relax.log') as f:
    ns_per_day = float([l for l in f if l.startswith('Performance:')][-1].split()[1])
production_ns = 1.0
print(f'{ns_per_day:.0f} ns/day: {production_ns} ns of production will take about {production_ns / ns_per_day * 24 * 60:.0f} minutes')

This is how you plan a simulation in practice: you measure the performance on a short run and then decide how long you can afford to simulate.

### Production

Now the system is ready for the production run, without position restraints. We simulate 1 ns, which is still very short: real studies use hundreds of nanoseconds or more. The production run continues the equilibration (`-t relax.cpt` passes the final positions and velocities), so in `5-npt.mdp` velocities are not generated again (`gen_vel = no`, `continuation = yes`).

In [ ]:
%cd {local_path}
%cp ../sb/Data/md/mdps/5-npt.mdp .
!grep -E "^(nsteps|dt|nstxout-compressed|compressed-x-grps|gen_vel|continuation) " 5-npt.mdp

In [ ]:
!gmx grompp -f 5-npt.mdp -c relax.gro -t relax.cpt -p topol.top -o npt.tpr

Colab can disconnect, for example if your laptop goes to sleep. For this reason we ask `mdrun` to write a checkpoint file every minute (`-cpt 1`) and to continue from it if it exists (`-cpi npt.cpt`). **If the run is interrupted**, reconnect, run again the installation cell and the cells that set `local_path` and move to it, and then run the next cell again: the simulation continues where it stopped instead of starting from zero.

In [ ]:
# run it: check the estimate above for how long it will take
!gmx mdrun -deffnm npt -cpi npt.cpt -cpt 1 -v

### Visualisation

Once the simulation has finished you should have a few more files:

In [ ]:
%ls npt*

*   `npt.xtc`: compressed trajectory file, here including only the protein (`compressed-x-grps = protein` in the .mdp file)
*   `npt.trr`: trajectory file including all atoms and velocities, written rarely
*   `npt.edr`: energy file with all the thermodynamic quantities
*   `npt.gro`: last frame of the trajectory
*   `npt.cpt`: checkpoint file for an exact restart
*   `npt.log`: log file

Remember the periodic boundary conditions? GROMACS keeps every atom inside the box, so a molecule that crosses one side of the box appears cut in two pieces, with very long bonds connecting them across the box. This is how the raw trajectory looks in VMD:

<img src="https://raw.githubusercontent.com/carlocamilloni/Structural-Bioinformatics/main/Data/md/images/lysozyme_vmd.png" width="400"/>

For visualisation and analysis we first make the protein whole again and keep it at the centre of the box:

In [ ]:
# -pbc mol: make molecules whole; -center: put the protein at the centre of the box;
# -ur compact: show the box in its most compact (spherical) shape
# we are asked for the group to center (Protein) and the group to write (Protein)
!printf "Protein\nProtein\n" | gmx trjconv -s npt.tpr -f npt.xtc -pbc mol -center -ur compact -o npt_whole.xtc
# the first frame, as a structure file containing the protein only
!echo Protein | gmx trjconv -s npt.tpr -f npt_whole.xtc -dump 0 -o protein_start.gro

To make a movie we also remove the overall rotation and translation of the protein by fitting every frame on the backbone of the first one, and we keep one frame every 20 ps (without hydrogens) to keep the file small:

In [ ]:
# we are asked for the group used for the fit (Backbone) and the group to write (Protein-H)
!printf "Backbone\nProtein-H\n" | gmx trjconv -s npt.tpr -f npt_whole.xtc -fit rot+trans -dt 20 -o movie.pdb

In [ ]:
import py3Dmol

view = py3Dmol.view(width=700, height=500)
view.addModelsAsFrames(open('movie.pdb').read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 100})
view.show()

You can also look at the trajectory with VMD on your laptop: download `protein_start.gro` and `npt_whole.xtc`, open `protein_start.gro` and then, from the menu `File`, choose *Load Data Into Molecule* and select `npt_whole.xtc`. This adds all the trajectory frames.

### Analysis

#### 1. Root Mean Square Deviation (RMSD)

The RMSD measures how far each frame is from a reference structure, after the best superposition. Here the reference is the energy-minimised crystal structure (`em.tpr`), and both the superposition and the RMSD use the backbone atoms.

GROMACS will warn you that the reference has many more atoms than the trajectory: this is expected, because the trajectory contains only the protein, which comes first in the system.

In [ ]:
# we are asked for the group used for the fit and the group used for the RMSD
!printf "Backbone\nBackbone\n" | gmx rms -s em.tpr -f npt_whole.xtc -o rmsd.xvg -tu ns

data, legends = read_xvg('rmsd.xvg')
plt.figure(figsize=(10, 5))
plt.plot(data[:, 0], data[:, 1])
plt.xlabel('Time (ns)')
plt.ylabel('Backbone RMSD (nm)')
plt.title('RMSD from the crystal structure')
plt.grid(True)
plt.show()

**Questions:** Does the RMSD reach a plateau? How large is it compared to the typical uncertainty of a crystal structure (a few tenths of an Å)? Would you consider the first part of the trajectory as part of the equilibration?

#### 2. Root Mean Square Fluctuations (RMSF)

The RMSF measures how much each atom fluctuates around its average position. We compute it for the C$\alpha$ atoms, i.e. one value per residue.

The crystal structure also contains information about fluctuations: the B-factors. For an atom fluctuating isotropically, $B = \frac{8\pi^2}{3}\,\mathrm{RMSF}^2$, so we can convert the C$\alpha$ B-factors of 1AKI into fluctuations and compare them with the simulation.

In [ ]:
!echo C-alpha | gmx rmsf -s em.tpr -f npt_whole.xtc -o rmsf.xvg -res

rmsf, _ = read_xvg('rmsf.xvg')
residues = rmsf[:, 0].astype(int)

# C-alpha B-factors from the crystal structure (columns 61-66 of the PDB format)
bfactor = {}
with open('1AKI_clean.pdb') as f:
    for line in f:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            bfactor[int(line[22:26])] = float(line[60:66])
b = np.array([bfactor[r] for r in residues])
rmsf_from_b = np.sqrt(3 * b / (8 * np.pi**2)) / 10   # from Å to nm

plt.figure(figsize=(12, 5))
plt.plot(residues, rmsf[:, 1], label='MD simulation')
plt.plot(residues, rmsf_from_b, label='from crystal B-factors')
plt.xlabel('Residue number')
plt.ylabel('C$\\alpha$ RMSF (nm)')
plt.legend()
plt.grid(True)
plt.show()

**Questions:** Which regions of the protein fluctuate more? Do they correspond to loops, helices or strands? Where do simulation and B-factors agree, and why would you not expect a perfect agreement? (Hint: what else contributes to the B-factors in a crystal, and how long is our simulation?)

#### 3. Radius of gyration

The radius of gyration measures how compact the protein is. Here we use a selection (`-sel Protein`) instead of answering a question; the result has one column for the total radius of gyration and three for its components around the x, y and z axes.

In [ ]:
!gmx gyrate -s npt.tpr -f npt_whole.xtc -sel Protein -o gyrate.xvg -tu ns

data, legends = read_xvg('gyrate.xvg')
plt.figure(figsize=(10, 5))
plt.plot(data[:, 0], data[:, 1])
plt.xlabel('Time (ns)')
plt.ylabel('Radius of gyration (nm)')
plt.grid(True)
plt.show()

#### 4. Secondary structure

`gmx dssp` assigns the secondary structure of every residue in every frame with the DSSP algorithm. It writes two files:

*   `dssp.dat`: one line per frame, one character per residue: `H` α-helix, `E` β-strand, `G` 3$_{10}$-helix, `I` π-helix, `P` polyproline II helix, `B` β-bridge, `T` turn, `S` bend, `~` loop, `=` chain break
*   `secondary.xvg` (from `-num`): for each frame, the number of residues in each type of secondary structure

In [ ]:
!gmx dssp -s npt.tpr -f npt_whole.xtc -sel Protein -o dssp.dat -num secondary.xvg -tu ns

data, legends = read_xvg('secondary.xvg')
# the columns are listed in the legends (the codes like \xa are Greek letters for the plotting program)
for i, name in enumerate(legends):
    print(i + 1, name)

In [ ]:
# column 10 = alpha-helices, column 8 = beta-strands (see the list above)
plt.figure(figsize=(10, 5))
plt.plot(data[:, 0], data[:, 10], label='α-helix')
plt.plot(data[:, 0], data[:, 8], label='β-strand')
plt.xlabel('Time (ns)')
plt.ylabel('Number of residues')
plt.title('Secondary structure content over time')
plt.legend()
plt.grid(True)
plt.show()

From `dssp.dat` we can also compute, for each residue, the fraction of frames in which it is in a helix or in a strand:

In [ ]:
with open('dssp.dat') as f:
    dssp = np.array([list(line.strip()) for line in f if line.strip()])
print(f'{dssp.shape[0]} frames, {dssp.shape[1]} residues')

helix_fraction = (dssp == 'H').mean(axis=0)
strand_fraction = (dssp == 'E').mean(axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(residues, helix_fraction, label='α-helix')
ax.bar(residues, strand_fraction, bottom=helix_fraction, label='β-strand')
ax.set_xlabel('Residue number')
ax.set_ylabel('Fraction of frames')
ax.set_title('Secondary structure per residue')
ax.legend()
plt.show()

**Question:** compare the RMSF and the secondary structure plots. Are the residues that fluctuate more also those without a stable secondary structure?

#### 5. Describing what you did

`gmx report-methods` summarises the simulation settings stored in a `.tpr` file. It is a good starting point for the methods section of a report, but it does not say everything: for example the force field, the water model and how the system was equilibrated are missing.

In [ ]:
!gmx report-methods -s npt.tpr -o methods.out > /dev/null 2>&1
!cat methods.out

More analysis tools are described in the GROMACS manual:
https://manual.gromacs.org/2026.4/user-guide/cmdline.html#commands-by-name